# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides step-by-step guidance for loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

First, list all record sets and their `@id`s, then list their fields/columns.

In [ ]:
# List all record sets
record_sets = dataset.metadata.record_sets
print("Available RecordSets:")
record_set_ids = []
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    record_set_ids.append(rs['@id'])

# For each record set, list fields
for rs in record_sets:
    print(f"\nFields in RecordSet '{rs['name']}' ({rs['@id']}):")
    for field in rs['fields']:
        print(f"  - {field['@id']} | name: {field['name']} | dataType: {field.get('dataType')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll load all available record sets into Pandas DataFrames.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet '@id': {record_set_id}")

# Example: Show columns from the first record set
if record_set_ids:
    primary_set_id = record_set_ids[0]
    print(f"\nColumns in RecordSet '@id': {primary_set_id}")
    print(dataframes[primary_set_id].columns.tolist())
    dataframes[primary_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We select numeric and grouping fields based on their `@id`. These can be found in the previous Data Overview output.

In [ ]:
# Example EDA on the first record set
record_set_id = primary_set_id
df = dataframes[record_set_id]

# Find numeric fields (e.g., GAD-7 scores, PHQ-9, PSQ)
numeric_field_ids = [col for col in df.columns if 'score' in col.lower() or col.lower() in ['gad_7_score', 'phq_9_score', 'psq_score']]
print(f"Numeric fields: {numeric_field_ids}")

# Use the first numeric field for demonstration
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    threshold = 10
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a demographic (e.g., 'gender' or similar field)
        group_field_candidates = [col for col in df.columns if 'gender' in col.lower() or 'education' in col.lower() or 'village' in col.lower()]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we produce a histogram for the numeric field, and, if available, a boxplot grouped by a demographic field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} ({record_set_id})")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot grouped by group_field, if present
if 'group_field' in locals() and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field} ({record_set_id})")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded mental health survey data from Kilifi County using the `mlcroissant` library, explored available record sets and fields by their `@id`, and performed exploratory analysis of numeric scores and their relationship to demographic fields.

- The dataset is structured using Croissant schema with clear record set and field identifiers.
- Numeric mental health scores such as GAD-7, PHQ-9, and PSQ were extracted and analyzed.
- Records were filtered, normalized, and grouped by demographic attributes (e.g., gender, education).
- Visualizations showed distributions and group differences, informing potential areas for deeper research.

Continue your exploration by referencing specific `@id` fields and record sets based on your analytical goals. See the metadata overview for all available entities.